In [10]:
# scripts/prepare_dataset.py
import os
import random
import pandas as pd
import numpy as np
import soundfile as sf
import json
from pathlib import Path
from typing import List, Dict

# ----------------------- 配置 -----------------------
ROOT = Path(r"D:\Project_Github\audio_click_mil")
AUDIO_DIR = ROOT / "data" / "original_audio"
CSV_DIR = ROOT / "data"

OUTPUT_ROOT = ROOT / "processed_data"
OUTPUT_ROOT.mkdir(exist_ok=True, parents=True)

TEST_FILES = set(range(1, 9))           # 1~8 作为独立测试集
EXCLUDE_FILES = {12}                    # file12 明确排除（无pulse信号）

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ----------------------- 工具函数 -----------------------
def get_audio_files() -> List[Path]:
    # 排除 file12
    all_files = [f for f in AUDIO_DIR.glob("*.wav") 
                 if parse_file_num(f.name) not in EXCLUDE_FILES]
    return sorted(all_files, key=lambda x: int(x.stem[-2:]))


def load_train_csvs():
    click = pd.read_csv(CSV_DIR / "ClickTrains.csv")
    burst = pd.read_csv(CSV_DIR / "BurstPulseTrains.csv")
    buzz = pd.read_csv(CSV_DIR / "BuzzTrains.csv")
    for df in [click, burst, buzz]:
        df.columns = df.columns.str.strip()
    return click, burst, buzz


def parse_file_num(filename: str) -> int:
    stem = Path(filename).stem
    return int(stem[-2:])


def get_audio_duration(audio_path: Path) -> float:
    info = sf.info(audio_path)
    return info.duration


# ----------------------- 1. 划分数据集 -----------------------
def split_dataset():
    all_files = get_audio_files()  # 已排除12
    file_nums = {parse_file_num(f.name): f for f in all_files}

    test_files = [file_nums[i] for i in range(1, 9) if i in file_nums]

    trainval_nums = [i for i in range(9, 36) if i not in EXCLUDE_FILES and i in file_nums]
    trainval_files = [file_nums[i] for i in trainval_nums]

    print(f"测试集文件数: {len(test_files)} (Ori 1-8)")
    print(f"训练/验证文件数: {len(trainval_files)} (已排除 file12，共 {len(trainval_nums)} 个)")

    random.shuffle(trainval_files)
    folds = np.array_split(trainval_files, 5)

    split_info = {
        "test": [str(p) for p in test_files],
        "folds": []
    }

    for i, fold_files in enumerate(folds, 1):
        val_file = fold_files[-1]
        train_files = fold_files[:-1]
        split_info["folds"].append({
            "fold": i,
            "train": [str(p) for p in train_files],
            "val": str(val_file)
        })

    with open(OUTPUT_ROOT / "data_split.json", "w", encoding="utf8") as f:
        json.dump(split_info, f, ensure_ascii=False, indent=2)

    print("划分完成，已保存到 processed_data/data_split.json")


# ----------------------- 2. 生成平衡bag标签（带补偿调整，全局平衡） -----------------------
def generate_balanced_bag_labels():
    click_df, burst_df, buzz_df = load_train_csvs()
    all_trains = pd.concat([click_df, burst_df, buzz_df], ignore_index=True)
    all_trains = all_trains[['Ori_file_num(No.)', 'Train_start(s)', 'Train_end(s)']]
    all_trains.columns = ['file_num', 'start', 'end']
    all_trains = all_trains.dropna()

    file_pos_records: Dict[int, list] = {}
    file_neg_dfs: Dict[int, pd.DataFrame] = {}

    print("\n=== 第一步：统计每个文件正负包数量（已排除 file12） ===")
    for audio_path in get_audio_files():
        file_num = parse_file_num(audio_path.name)
        duration = get_audio_duration(audio_path)

        bag_duration = 60.0
        num_bags = int(np.ceil(duration / bag_duration))

        pos_records = []
        neg_bags = []

        for bag_idx in range(num_bags):
            bag_start = bag_idx * bag_duration
            bag_end = min((bag_idx + 1) * bag_duration, duration)

            overlapping = all_trains[
                (all_trains['file_num'] == file_num) &
                (all_trains['end'] > bag_start) &
                (all_trains['start'] < bag_end)
            ]

            is_positive = len(overlapping) > 0
            total_train_sec = sum(
                min(row['end'], bag_end) - max(row['start'], bag_start)
                for _, row in overlapping.iterrows()
            ) if is_positive else 0.0

            bag_info = {
                'file_num': file_num,
                'audio_path': str(audio_path),
                'bag_idx': bag_idx,
                'bag_start_sec': bag_start,
                'bag_end_sec': bag_end,
                'bag_label': 1 if is_positive else 0,
                'total_train_duration_sec': round(total_train_sec, 4),
                'bag_duration_sec': round(bag_end - bag_start, 4)
            }

            if is_positive:
                pos_records.append(bag_info)
            else:
                neg_bags.append(bag_info)

        file_pos_records[file_num] = pos_records
        if neg_bags:
            df_neg = pd.DataFrame(neg_bags)
            df_neg['signal_ratio'] = 0.0
            df_neg['noise_ratio'] = 1.0
            file_neg_dfs[file_num] = df_neg

        print(f"文件 {file_num:02d}: 正包 {len(pos_records):3d} 个 | 负包 {len(neg_bags):3d} 个")

    total_pos = sum(len(recs) for recs in file_pos_records.values())
    print(f"\n全局总正包数量: {total_pos}")

    # 初始目标负包数（按正包比例）
    target_neg = {fn: round(len(file_pos_records[fn]) / total_pos * total_pos) 
                  for fn in file_pos_records}

    # 计算短缺和剩余，并进行补偿调整
    shortage = 0
    surplus_files = []
    for fn in file_pos_records:
        avail = len(file_neg_dfs.get(fn, pd.DataFrame()))
        tgt = target_neg[fn]
        if avail < tgt:
            shortage += (tgt - avail)
            target_neg[fn] = avail
        else:
            surplus_files.append((fn, avail - tgt))

    # 把短缺分配给有剩余的文件（按剩余容量比例）
    if shortage > 0 and surplus_files:
        total_surplus_capacity = sum(s[1] for s in surplus_files)
        if total_surplus_capacity > 0:
            for fn, extra in surplus_files:
                add = round(shortage * extra / total_surplus_capacity)
                target_neg[fn] += add
                # 防止超过可用数量
                if target_neg[fn] > len(file_neg_dfs.get(fn, pd.DataFrame())):
                    target_neg[fn] = len(file_neg_dfs.get(fn, pd.DataFrame()))

    print("\n=== 第二步：采样负包（带补偿调整） ===")
    balanced_neg_records = []

    for fn in file_pos_records:
        pos_count = len(file_pos_records[fn])
        tgt = target_neg.get(fn, 0)
        df_neg = file_neg_dfs.get(fn, pd.DataFrame())
        avail = len(df_neg)

        n_take = min(tgt, avail)
        if n_take > 0:
            sampled = df_neg.sample(n=n_take, random_state=RANDOM_SEED)
            balanced_neg_records.extend(sampled.to_dict('records'))

        print(f"文件 {fn:02d}: 正包 {pos_count:3d} | 目标负包 {tgt:3d} | 可用 {avail:3d} | 实际选取 {n_take:3d}")

    # 合并正包和负包
    all_pos = []
    for recs in file_pos_records.values():
        for r in recs:
            r['signal_ratio'] = r['total_train_duration_sec'] / r['bag_duration_sec']
            r['noise_ratio'] = 1.0 - r['signal_ratio']
            all_pos.append(r)

    df_balanced = pd.concat([pd.DataFrame(all_pos), pd.DataFrame(balanced_neg_records)], ignore_index=True)

    # 保存
    output_csv = OUTPUT_ROOT / "balanced_bag_labels.csv"
    df_balanced.to_csv(output_csv, index=False, encoding='utf-8')

    final_pos = (df_balanced['bag_label'] == 1).sum()
    final_neg = len(df_balanced) - final_pos
    print("\n=== 全局平衡结果 ===")
    print(f"最终正包数量: {final_pos}")
    print(f"最终负包数量: {final_neg}")
    print(f"最终正包比例: {final_pos / len(df_balanced) * 100:.1f}%")
    print(f"平衡数据集已保存: {output_csv}")

    # 正包噪声统计
    pos_bags = df_balanced[df_balanced['bag_label'] == 1]
    noise_pct = pos_bags['noise_ratio'] * 100
    stats_text = f"""正包噪声成分占比统计（仅正包，单位：%）：
范围: {noise_pct.min():.2f} - {noise_pct.max():.2f}
均值: {noise_pct.mean():.2f}
方差: {noise_pct.var():.2f}
中位数 (50%): {np.percentile(noise_pct, 50):.2f}
第1/5位 (20%): {np.percentile(noise_pct, 20):.2f}
第2/5位 (40%): {np.percentile(noise_pct, 40):.2f}
第4/5位 (80%): {np.percentile(noise_pct, 80):.2f}
"""
    stats_txt = OUTPUT_ROOT / "noise_stats.txt"
    with open(stats_txt, "w", encoding="utf8") as f:
        f.write(stats_text)

    print("\n" + stats_text)
    print(f"正包噪声统计已保存到: {stats_txt}")


if __name__ == "__main__":
    print("开始执行数据准备与划分...")
    split_dataset()
    print("\n" + "="*60)
    print("开始生成平衡bag标签（已明确排除 file12，带补偿调整保证全局平衡）...")
    generate_balanced_bag_labels()
    print("所有数据准备步骤完成！")

开始执行数据准备与划分...
测试集文件数: 8 (Ori 1-8)
训练/验证文件数: 26 (已排除 file12，共 26 个)
划分完成，已保存到 processed_data/data_split.json

开始生成平衡bag标签（已明确排除 file12，带补偿调整保证全局平衡）...

=== 第一步：统计每个文件正负包数量（已排除 file12） ===
文件 01: 正包   7 个 | 负包  23 个
文件 02: 正包  21 个 | 负包   9 个
文件 03: 正包   5 个 | 负包  25 个
文件 04: 正包   6 个 | 负包  24 个
文件 05: 正包  11 个 | 负包  19 个
文件 06: 正包  16 个 | 负包  14 个
文件 07: 正包   6 个 | 负包  24 个
文件 08: 正包  12 个 | 负包  18 个
文件 09: 正包  19 个 | 负包  11 个
文件 10: 正包  19 个 | 负包  11 个
文件 11: 正包   4 个 | 负包   9 个
文件 13: 正包   8 个 | 负包  22 个
文件 14: 正包  13 个 | 负包  17 个
文件 15: 正包   3 个 | 负包  15 个
文件 16: 正包   7 个 | 负包  23 个
文件 17: 正包  15 个 | 负包  15 个
文件 18: 正包   4 个 | 负包  26 个
文件 19: 正包   8 个 | 负包  22 个
文件 20: 正包   5 个 | 负包  25 个
文件 21: 正包  16 个 | 负包  14 个
文件 22: 正包  18 个 | 负包  12 个
文件 23: 正包   9 个 | 负包  21 个
文件 24: 正包  13 个 | 负包  17 个
文件 25: 正包  11 个 | 负包  19 个
文件 26: 正包   3 个 | 负包  27 个
文件 27: 正包   2 个 | 负包  28 个
文件 28: 正包   7 个 | 负包  23 个
文件 29: 正包  18 个 | 负包  12 个
文件 30: 正包  12 个 | 负包  18 个
文件 31: 正包   8 个 | 负包  22 个
文件